# Building LLM

## Goal

This notebook is a hands-on journey to build a language model from scratch.

Each version introduces one new concept, allowing the model to evolve step by step while practicing language-model development.

---

## Version 2

In this version, we extend the character model with a small neural language model implemented directly in NumPy.

The model reuses the same training text, vocabulary, and adjacent character pairs from Version 1, but replaces direct transition counting with a trainable weight matrix, softmax probabilities, cross-entropy loss, and gradient descent.

This version keeps the context and generation process unchanged, making the transition from statistical counts to neural parameter learning simple and visible.

## 1. Imports

In [1]:
import numpy as np

## 2. Training Data

### Training text

The English corpus is included directly in the notebook. It provides the text from which the model learns character transitions.

In [2]:
corpus = 'language models learn patterns from text.\na small model predicts what character may come next.\nwe begin with counting because counting is easy to inspect.\nthe model sees letters, spaces, and punctuation.\neach prediction comes from examples found in the training text.\nsimple systems help us understand more advanced systems.\nlater versions will learn parameters with neural networks.\nclear experiments make machine learning easier to study.'

print(corpus)
print("Characters:", len(corpus))

language models learn patterns from text.
a small model predicts what character may come next.
we begin with counting because counting is easy to inspect.
the model sees letters, spaces, and punctuation.
each prediction comes from examples found in the training text.
simple systems help us understand more advanced systems.
later versions will learn parameters with neural networks.
clear experiments make machine learning easier to study.
Characters: 440


### Vocabulary

The vocabulary is the set of symbols the model can represent. Because this is a character model, every letter, space, punctuation mark and newline is a token.
The neural model also assigns an integer identifier to every character so that characters can be represented numerically.

In [3]:
vocabulary = sorted(set(corpus))
vocabulary_size = len(vocabulary)

character_to_id = {character: index for index, character in enumerate(vocabulary)}
id_to_character = {index: character for character, index in character_to_id.items()}

print("Vocabulary size:", vocabulary_size)
print("Vocabulary:", repr("".join(vocabulary)))
print("First mappings:", list(character_to_id.items())[:10])

Vocabulary size: 27
Vocabulary: '\n ,.abcdefghiklmnoprstuvwxy'
First mappings: [('\n', 0), (' ', 1), (',', 2), ('.', 3), ('a', 4), ('b', 5), ('c', 6), ('d', 7), ('e', 8), ('f', 9)]


### Character pairs and numerical encoding

Each character is converted into its integer identifier.

For the text `model`:

`model`  
↓  
`[15, 17, 7, 8, 14]`

Adjacent identifiers form the training examples:

- `m -> o` becomes `15 -> 17`
- `o -> d` becomes `17 -> 7`
- `d -> e` becomes `7  -> 8`
- `e -> l` becomes `8  -> 14`

The first identifier is the input. The following identifier is the target to predict.

During generation, predicted identifiers are converted back into characters:

`[15, 17, 7, 8, 14]`  
↓  
`model`

In [4]:
examples = list(zip(corpus, corpus[1:]))

input_ids = np.array([character_to_id[current_character] for current_character, _ in examples])
target_ids = np.array([character_to_id[next_character] for _, next_character in examples])

print("Number of examples:", len(examples))
print("First 12 examples:", examples[:12])
print("First 12 input IDs:", input_ids[:12])
print("First 12 target IDs:", target_ids[:12])

Number of examples: 439
First 12 examples: [('l', 'a'), ('a', 'n'), ('n', 'g'), ('g', 'u'), ('u', 'a'), ('a', 'g'), ('g', 'e'), ('e', ' '), (' ', 'm'), ('m', 'o'), ('o', 'd'), ('d', 'e')]
First 12 input IDs: [14  4 16 10 22  4 10  8  1 15 17  7]
First 12 target IDs: [ 4 16 10 22  4 10  8  1 15 17  7  8]


## 3. Neural Model

### One-hot inputs and trainable parameters

Each input character is represented by a one-hot vector. A one-hot vector contains one value equal to 1 and all other values equal to 0.

Multiplying this representation by the weight matrix produces one score for every possible next character. These scores are called logits.

In [5]:
inputs = np.eye(vocabulary_size)[input_ids]

model_random = np.random.default_rng(42)

weights = model_random.normal(loc=0.0, scale=0.01, size=(vocabulary_size, vocabulary_size))

print("Input matrix shape:", inputs.shape)
print("Weight matrix shape:", weights.shape)
print("Trainable parameters:", weights.size)

Input matrix shape: (439, 27)
Weight matrix shape: (27, 27)
Trainable parameters: 729


### Softmax probabilities

The weight matrix produces logits, which are unrestricted numerical scores.

Softmax converts the logits into probabilities:

- every probability is between 0 and 1;
- the probabilities for one input sum to 1;
- higher logits produce higher probabilities.

Subtracting the maximum logit before exponentiation improves numerical stability.

In [6]:
def softmax(logits):
    shifted_logits = logits - np.max(logits, axis=1, keepdims=True)

    exponentials = np.exp(shifted_logits)

    return exponentials / exponentials.sum(axis=1, keepdims=True)

### Cross-entropy loss

Cross-entropy measures how much probability the model assigns to the correct next character.

A high probability for the correct character produces a low loss. A low probability produces a high loss.

Training updates the weights to reduce this value.

In [7]:
def calculate_loss(weights, inputs, targets):
    logits = inputs @ weights
    probabilities = softmax(logits)

    correct_probabilities = probabilities[np.arange(len(targets)), targets]

    return -np.mean(np.log(correct_probabilities + 1e-12))

In [8]:
initial_loss = calculate_loss(weights, inputs, target_ids)

print("Initial loss:", initial_loss)

Initial loss: 3.2972712476038026


### Training with gradient descent

Training repeatedly:

1. calculates the predictions;
2. measures the loss;
3. calculates the gradients;
4. updates the weights.

All examples are processed together at every step. The gradients are implemented directly with NumPy, without automatic differentiation.

In [9]:
def train_model(inputs, targets, initial_weights, learning_rate=20.0, steps=1000):
    trained_weights = initial_weights.copy()
    loss_history = []

    for step in range(steps + 1):
        logits = inputs @ trained_weights
        probabilities = softmax(logits)

        correct_probabilities = probabilities[np.arange(len(targets)), targets]

        loss = -np.mean(np.log(correct_probabilities + 1e-12))

        if step % 100 == 0:
            loss_history.append(loss)
            print(f"Step {step:4d} | Loss: {loss:.4f}")

        if step == steps:
            break

        logits_gradient = probabilities.copy()

        logits_gradient[np.arange(len(targets)), targets] -= 1

        logits_gradient /= len(targets)

        weights_gradient = inputs.T @ logits_gradient

        trained_weights -= (learning_rate * weights_gradient)

    return trained_weights, loss_history

In [10]:
weights, loss_history = train_model(
    inputs,
    target_ids,
    weights,
    learning_rate=20.0,
    steps=1000
)

final_loss = calculate_loss(
    weights,
    inputs,
    target_ids
)

print("Initial loss:", initial_loss)
print("Final loss:", final_loss)

Step    0 | Loss: 3.2973
Step  100 | Loss: 2.0113
Step  200 | Loss: 1.9727
Step  300 | Loss: 1.9611
Step  400 | Loss: 1.9555
Step  500 | Loss: 1.9523
Step  600 | Loss: 1.9502
Step  700 | Loss: 1.9487
Step  800 | Loss: 1.9476
Step  900 | Loss: 1.9468
Step 1000 | Loss: 1.9461
Initial loss: 3.2972712476038026
Final loss: 1.9461102216662014


### Learned probabilities

After training, the model can produce a next-character probability distribution for any character in the vocabulary.

The probabilities now come from the trained weights instead of direct transition counts.

In [11]:
def next_character_probabilities(weights, current_character):
    if current_character not in character_to_id:
        raise ValueError("The character is not in the vocabulary.")

    current_id = character_to_id[current_character]
    logits = weights[current_id:current_id + 1]
    probabilities = softmax(logits)[0]

    return {
        id_to_character[index]: probability
        for index, probability in enumerate(probabilities)
    }

In [12]:
probabilities_after_m = next_character_probabilities(weights, "m")

most_likely_after_m = sorted(
    probabilities_after_m.items(),
    key=lambda item: item[1],
    reverse=True
)[:10]

print("Most likely characters after 'm':")

for character, probability in most_likely_after_m:
    print(f"{repr(character):>4}: {probability:.4f}")

print("Total:", sum(probabilities_after_m.values()))

Most likely characters after 'm':
 'a': 0.2212
 'e': 0.2212
 'o': 0.2212
 ' ': 0.1101
 'p': 0.1101
 's': 0.1101
 'u': 0.0003
 'l': 0.0003
 'i': 0.0003
 'h': 0.0003
Total: 0.9999999999999997


## 4. Generator

### Sample the next character

The current character selects one row of trained logits.

Softmax converts the logits into probabilities, and the next character is sampled from that distribution.

The sampled character becomes the new input for the following step.

In [13]:
def sample_next_character(weights, current_id, random_generator):
    logits = weights[current_id:current_id + 1]
    probabilities = softmax(logits)[0]

    return random_generator.choice(vocabulary_size, p=probabilities)

In [14]:
demo_random = np.random.default_rng(42)
m_id = character_to_id["m"]

sampled_ids = [sample_next_character(weights, m_id, demo_random) for _ in range(5)]

print("Five samples after 'm':", [id_to_character[index] for index in sampled_ids])

Five samples after 'm': ['o', 'e', 'p', 'o', ' ']


### Generate text

Generation repeats the same loop:

1. read the current character;
2. sample the next character;
3. append it to the output;
4. use the new character as the next context.

The seed makes the example reproducible.

The probabilities are produced by the trained parameters instead of being calculated directly from transition counts.

In [15]:
def generate_text(weights, start_character="i", length=200, seed=42):
    if length < 1:
        raise ValueError("Length must be at least 1.")

    if start_character not in character_to_id:
        raise ValueError("The start character is not in the vocabulary.")

    random_generator = np.random.default_rng(seed)

    generated = [start_character]
    current_id = character_to_id[start_character]

    for _ in range(length - 1):
        current_id = sample_next_character(weights, current_id, random_generator)

        generated.append(id_to_character[current_id])

    return "".join(generated)

In [16]:
generated_text = generate_text(weights, start_character="i", length=300, seed=42)

print(generated_text)

iompl wiodedystear tiomstran cors leacodinsmats, s e tining comorng icmampach plledil cth mared pbe wonctrneacth in.
tomong m macer veco wio mexth ine tt.
ararniexp ndioms mauaclstouseasin s wh tadeteasy.
aier mon.
spacteangunt ng clsits.
acts moundeers.
cts be te vake ace pleanst.
chee le t.
attwio


## 5. Tests

These assertions verify the numerical representation, the neural model, the training result and the reproducibility of generation.

In [17]:
assert len(examples) == len(corpus) - 1
assert len(input_ids) == len(examples)
assert len(target_ids) == len(examples)
assert inputs.shape == (len(examples), vocabulary_size)
assert weights.shape == (vocabulary_size, vocabulary_size)
assert np.allclose(inputs.sum(axis=1), 1.0)
assert final_loss < initial_loss
assert abs(sum(next_character_probabilities(weights, "m").values()) - 1.0) < 1e-12
assert generate_text(weights, "l", 30, seed=10) == generate_text(weights, "l", 30, seed=10)
assert len(generate_text(weights, "l", 30, seed=10)) == 30

print("All checks passed.")

All checks passed.


## Notes

- Characters are converted into integer identifiers and one-hot vectors.
- A trainable weight matrix replaces direct transition counts.
- Softmax converts the model logits into next-character probabilities.
- Cross-entropy measures the prediction error.
- Gradient descent updates the weights and reduces the loss.
- The generator remains autoregressive and samples one character at a time.
- The tests verify the numerical representation, probability distributions, training result and reproducibility.

Future versions will introduce new components and gradually evolve the architecture.